In [0]:
spark.conf.set("fs.azure.account.auth.type.proddatalakefordatabrick.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.proddatalakefordatabrick.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.proddatalakefordatabrick.dfs.core.windows.net", "8b03b30e-c838-4568-a41e-a801634f0e96")
spark.conf.set("fs.azure.account.oauth2.client.secret.proddatalakefordatabrick.dfs.core.windows.net", dbutils.secrets.get(scope="darshanscope", key="app-secret"))
spark.conf.set("fs.azure.account.oauth2.client.endpoint.proddatalakefordatabrick.dfs.core.windows.net", "https://login.microsoftonline.com/5ff33f54-37cd-4d6d-9156-f6c00f164999/oauth2/token")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp

df_silver = spark.read.table("proddatabricksdemo.silver.customer")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS proddatabricksdemo.gold.DimCustomer (
    DimCustomerKey BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id STRING,
    first_name STRING,
    last_name STRING,
    email STRING,
    city STRING,
    state STRING,
    Domain STRING,
    create_date TIMESTAMP,
    update_date TIMESTAMP,
    full_name STRING
)
USING DELTA;

In [0]:
# Target Gold Delta Table
gold_table = DeltaTable.forName(
    spark,
    "proddatabricksdemo.gold.DimCustomer"
)

In [0]:
(
    gold_table.alias("target")
    .merge(
        df_silver.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdate(
        set={
            "first_name": "source.full_name",
            "last_name": "source.full_name",
            "email": "source.email",
            "city": "source.city",
            "state": "source.state",
            "Domain": "source.domain",
            "update_date": "current_timestamp()",
            "full_name": "source.full_name"
        }
    )
    .whenNotMatchedInsert(
        values={
            "customer_id": "source.customer_id",
            "first_name": "source.first_name",
            "last_name": "source.last_name",
            "email": "source.email",
            "city": "source.city",
            "state": "source.state",
            "Domain": "source.domain",
            "create_date": "current_timestamp()",
            "update_date": "current_timestamp()",
            "full_name": "source.full_name"
        }
    )
    .execute()
)

In [0]:
df_final = spark.read.table("proddatabricksdemo.gold.DimCustomer")
df_final.display

In [0]:
df_final.write.format("delta").mode("overwrite").save("abfss://destination@proddatalakefordatabrick.dfs.core.windows.net/gold/customer")